# Zapis modeli do pliku, ponowne wczytanie i użycie do predykcji.

* Jeden model to zwykły `RandomForestClassifier`, a drugi to `Pipeline`: `SimpleImputer` + `StandardScaler` + `LogisticRegression`. `Pipeline` w `sklearn` pozwala połączyć kolejne etapy przetwarzania danych z modelem końcowym, więc po zapisujemy od razu także preprocessing, nie tylko sam klasyfikator. 
* Do zapisu używam `joblib`, które jest typowym sposobem zapisu modeli `scikit-learn`. 

In [1]:
import pandas as pd

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import numpy as np
import matplotlib.pyplot as plt

from joblib import dump, load

# Wczytanie danych

In [2]:
dane = load_breast_cancer(as_frame=True)

X = dane.data
y = dane.target

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Model 1: Random Forest

In [3]:
random_forest_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

random_forest_model.fit(X_train, y_train)

rf_pred = random_forest_model.predict(X_test)

print("Random Forest accuracy:", accuracy_score(y_test, rf_pred))

Random Forest accuracy: 0.956140350877193


# Model 2: regresja logistyczna w Pipeline

Tutaj mamy trzy kroki:

1. `SimpleImputer` — uzupełnia braki danych medianą.
2. `StandardScaler` — standaryzuje zmienne.
3. `LogisticRegression` — właściwy model klasyfikacyjny.

In [4]:
logistic_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=5000))
])

logistic_pipeline.fit(X_train, y_train)

logreg_pred = logistic_pipeline.predict(X_test)

print("Logistic Regression Pipeline accuracy:", accuracy_score(y_test, logreg_pred))

Logistic Regression Pipeline accuracy: 0.9824561403508771


# Zapis modeli do plików

In [5]:
dump(random_forest_model, "random_forest_model.joblib")

['random_forest_model.joblib']

In [6]:
dump(logistic_pipeline, "logistic_pipeline.joblib")

['logistic_pipeline.joblib']

# Ponowne załadowanie modeli

In [7]:
loaded_random_forest = load("random_forest_model.joblib")
loaded_logistic_pipeline = load("logistic_pipeline.joblib")

# Użycie załadowanych modeli

In [8]:
nowe_dane = X_test.iloc[:5]

rf_predictions = loaded_random_forest.predict(nowe_dane)
logreg_predictions = loaded_logistic_pipeline.predict(nowe_dane)

print("Predykcje Random Forest:")
print(rf_predictions)

print("Predykcje Logistic Regression Pipeline:")
print(logreg_predictions)

Predykcje Random Forest:
[0 1 0 0 0]
Predykcje Logistic Regression Pipeline:
[0 1 0 1 0]


In [9]:
etykiety_klas = {
    0: "nowotwór złośliwy",
    1: "nowotwór łagodny"
}

In [10]:
rf_predictions_text = [etykiety_klas[pred] for pred in rf_predictions]
logreg_predictions_text = [etykiety_klas[pred] for pred in logreg_predictions]

In [11]:
print("Predykcje Random Forest:")
print(rf_predictions_text)

print("Predykcje Logistic Regression Pipeline:")
print(logreg_predictions_text)

Predykcje Random Forest:
['nowotwór złośliwy', 'nowotwór łagodny', 'nowotwór złośliwy', 'nowotwór złośliwy', 'nowotwór złośliwy']
Predykcje Logistic Regression Pipeline:
['nowotwór złośliwy', 'nowotwór łagodny', 'nowotwór złośliwy', 'nowotwór łagodny', 'nowotwór złośliwy']


# Predykcja prawdopodobieństw

In [12]:
rf_probabilities = loaded_random_forest.predict_proba(nowe_dane)
logreg_probabilities = loaded_logistic_pipeline.predict_proba(nowe_dane)

In [13]:
tabela_predykcji = pd.DataFrame({
        "rf_predictions": rf_predictions,
        "logreg_predictions": logreg_predictions,
        "rf_prawdopodobienstwo_klasy_1": np.round(rf_probabilities[:, 1], 2),
        "logreg_prawdopodobienstwo_klasy_1": np.round(logreg_probabilities[:, 1], 2),
        "roznica_prawdopodobienstw": np.round(
            np.abs(rf_probabilities[:, 1] - logreg_probabilities[:, 1]), 2
        ),
        "zgodnosc_modeli": rf_predictions == logreg_predictions
})

In [14]:
tabela_predykcji

,rf_predictions,logreg_predictions,rf_prawdopodobienstwo_klasy_1,logreg_prawdopodobienstwo_klasy_1,roznica_prawdopodobienstw,zgodnosc_modeli
0,0,0,0.00,0.00,0.00,True
1,1,1,1.00,1.00,0.00,True
2,0,0,0.12,0.01,0.11,True
3,0,1,0.28,0.53,0.25,False
4,0,0,0.01,0.00,0.01,True


# Bibliografia:
* https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html
* https://scikit-learn.org/stable/model_persistence.html